# 02 — Dynamic Network Science, Econometric Spillovers & Propagation Baselines

### Academic Research Objectives:
1. **Multi-Network Topologies:** Construct and compare four dynamic network representations (strictly using trailing windows):
   - **Correlation Network:** Undirected market dependence.
   - **Minimum Spanning Tree (MST):** Ultrametric space and topological backbone (Mantegna & Onnela).
   - **Directed Lead-Lag Network:** High-frequency temporal lead-lag influence.
   - **Econometric Spillover Network:** Diebold-Yilmaz (2012) Generalized Forecast Error Variance Decomposition (GFEVD) across the 30 coins.
2. **Centrality & Influence Dynamics:** Track Degree, Betweenness, Eigenvector, and PageRank over time.
3. **Community Structure & Modularity:** Seeded Louvain community detection, modularity tracking across historical events, and Normalized Mutual Information (NMI) against external sector taxonomies.
4. **Crisis Topological Contraction:** Evaluate MST normalized tree length $L(t)$ around systemic crises (e.g. August 2024 Yen carry unwind, FTX crash).
5. **Propagation Baseline (Independent Cascade):** Empirical diffusion benchmark fitted on training cascades and evaluated via AUROC and AUPRC on validation/test partitions.
6. **Artifact Generation:** Verify `rankings.csv`, `network_features.pt`, and `cascade_baseline.csv`.


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Add src to path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import normalized_mutual_info_score

from cryptograph.data.universe import TARGET_UNIVERSE_30, SECTOR_TAXONOMY
from cryptograph.analysis.connectedness import VARSpilloverBaseline
from cryptograph.analysis.lead_lag import compute_lead_lag_strength_matrix
from cryptograph.network.centrality import NetworkCentralityAnalyzer
from cryptograph.network.communities import CommunityDetector
from cryptograph.network.mst import MinimumSpanningTreeAnalyzer
from cryptograph.network.topology import TopologyMetrics
from cryptograph.network.cascade import IndependentCascadeModel


## 1. Load Processed 30-Coin Hourly Dataset & External Events Catalogue

We load the unified 30-coin hourly dataset (`data/processed/v2/1h/processed_dataset.pt`), verified sectors (`data/external/sectors.csv`), and external crisis catalog (`data/external/events.csv`).


In [2]:
dataset_path = Path("../data/processed/v2/1h/processed_dataset.pt")
data = torch.load(dataset_path, map_location="cpu", weights_only=False)

asset_names = data["asset_names"]
timestamps = pd.to_datetime(data["timestamps"])
close_prices = pd.DataFrame(data["close_prices"].numpy(), index=timestamps, columns=asset_names)
returns = np.log(close_prices / close_prices.shift(1)).dropna()

# Load sector taxonomy
sectors_df = pd.read_csv("../data/external/sectors.csv")
sector_map = dict(zip(sectors_df["coin"], sectors_df["sector"]))

# Load historical crises catalogue
events_df = pd.read_csv("../data/external/events.csv")
events_df["date"] = pd.to_datetime(events_df["date"])

print(f"Loaded {len(asset_names)} cryptocurrencies across {len(timestamps):,} hourly timesteps.")
print(f"Sectors: {len(set(sector_map.values()))} unique taxonomy classes.")
print(f"Documented Historical Crises: {len(events_df)} benchmark events.")


Loaded 30 cryptocurrencies across 24,024 hourly timesteps.
Sectors: 10 unique taxonomy classes.
Documented Historical Crises: 11 benchmark events.


## 2. Multi-Network Topologies: Correlation, MST, Lead-Lag, and DY-GFEVD

We construct four distinct graph representations over trailing windows:
1. **Correlation Graph ($G_C$):** Undirected edges formed when $|\rho_{ij}| \ge 0.40$.
2. **Minimum Spanning Tree ($G_{MST}$):** Connected tree on $N-1$ edges minimizing Mantegna metric distance $d_{ij} = \sqrt{2(1 - \rho_{ij})}$.
3. **Directed Lead-Lag Network ($G_{LL}$):** Weighted directed edges $L_{i \to j} = \max_{\tau \in [1, 4]} \rho_{ij}(\tau) - \max_{\tau \in [1, 4]} \rho_{ij}(-\tau)$.
4. **Econometric Spillover Network ($G_{DY}$):** Directed weighted edges $\theta_{ij}^g(H)$ from order-invariant VAR-GFEVD.


In [3]:
# Use a representative 720-hour (30-day) trailing window for topological visualization
sample_ret = returns.iloc[-720:]

# 1. Correlation Network
corr_matrix = sample_ret.corr().abs()
G_corr = nx.Graph()
for a in asset_names:
    G_corr.add_node(a, sector=sector_map.get(a, "Other"))
for i in range(len(asset_names)):
    for j in range(i + 1, len(asset_names)):
        c = float(corr_matrix.iloc[i, j])
        if c >= 0.40:
            G_corr.add_edge(asset_names[i], asset_names[j], weight=c)

# 2. Minimum Spanning Tree
G_mst, norm_tree_len = MinimumSpanningTreeAnalyzer.build_mst(corr_matrix)

# 3. Directed Lead-Lag Network
lead_matrix = compute_lead_lag_strength_matrix(sample_ret, max_lag=4)
G_lead = nx.DiGraph()
for a in asset_names:
    G_lead.add_node(a, sector=sector_map.get(a, "Other"))
for u in asset_names:
    for v in asset_names:
        w = float(lead_matrix.loc[u, v])
        if w > 0.02:
            G_lead.add_edge(u, v, weight=w)

# 4. Diebold-Yilmaz Spillover Matrix
var_model = VARSpilloverBaseline(maxlags=1, forecast_horizon=10)
var_model.fit(sample_ret)
spillover_res = var_model.compute_spillover_table()
spill_df = spillover_res["directional_table"]

print(f"Graph Construction Summary (Trailing 30-Day Window):")
print(f"  Correlation Graph (threshold=0.40): {G_corr.number_of_nodes()} nodes, {G_corr.number_of_edges()} edges")
print(f"  Minimum Spanning Tree (MST):         {G_mst.number_of_nodes()} nodes, {G_mst.number_of_edges()} edges (Norm Len: {norm_tree_len:.3f})")
print(f"  Directed Lead-Lag Graph:             {G_lead.number_of_nodes()} nodes, {G_lead.number_of_edges()} edges")
print(f"  Total Spillover Index (TSI):         {spillover_res['total_spillover_index']:.2f}%")


[2026-10-02 17:27:47] [INFO] [cryptograph.analysis.connectedness] - Fitted VAR baseline with lag order: 1


Graph Construction Summary (Trailing 30-Day Window):
  Correlation Graph (threshold=0.40): 30 nodes, 338 edges
  Minimum Spanning Tree (MST):         30 nodes, 29 edges (Norm Len: 0.798)
  Directed Lead-Lag Graph:             30 nodes, 248 edges
  Total Spillover Index (TSI):         87.59%


C:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


## 3. Centrality Dynamics: PageRank, Degree, and Eigenvector

We compute centrality metrics across both undirected and directed topologies:
- **PageRank:** Applied to directed networks ($G_{LL}$ and $G_{DY}$) to determine persistent information sinks and sources.
- **Betweenness & Eigenvector:** Applied to the correlation network to identify bridge assets and core cluster anchors.


In [4]:
cent_corr = NetworkCentralityAnalyzer.compute_all_centralities(G_corr)
lead_pagerank = nx.pagerank(G_lead, alpha=0.85, weight="weight") if G_lead.number_of_edges() > 0 else {a: 1/len(asset_names) for a in asset_names}

centrality_comparison = pd.DataFrame({
    "Sector": [sector_map.get(a, "Other") for a in asset_names],
    "Degree": [cent_corr.loc[a, "degree"] if a in cent_corr.index else 0.0 for a in asset_names],
    "Eigenvector": [cent_corr.loc[a, "eigenvector"] if a in cent_corr.index else 0.0 for a in asset_names],
    "Betweenness": [cent_corr.loc[a, "betweenness"] if a in cent_corr.index else 0.0 for a in asset_names],
    "LeadLag_PageRank": [lead_pagerank.get(a, 0.0) for a in asset_names],
    "DY_Net_Spillover": [spillover_res["net_spillover"].get(a, 0.0) for a in asset_names]
}, index=asset_names)

print("Top 10 Assets Ranked by Lead-Lag PageRank:")
print(centrality_comparison.sort_values("LeadLag_PageRank", ascending=False).head(10)[["Sector", "LeadLag_PageRank", "Degree", "DY_Net_Spillover"]])

fig_cent = px.scatter(
    centrality_comparison.reset_index(),
    x="LeadLag_PageRank", y="DY_Net_Spillover", color="Sector",
    text="index", size="Degree",
    title="Centrality Alignment: Directed Lead-Lag PageRank vs Econometric NET Spillover",
    labels=dict(index="Coin", LeadLag_PageRank="PageRank (Lead-Lag)", DY_Net_Spillover="DY NET Spillover (%)")
)
fig_cent.update_traces(textposition="top center")
fig_cent.update_layout(template="plotly_dark", height=500)
fig_cent.show()


Top 10 Assets Ranked by Lead-Lag PageRank:


                Sector  LeadLag_PageRank    Degree  DY_Net_Spillover
ARB            Layer 2          0.176660  0.206897        -36.599462
FIL              DePIN          0.174874  0.724138        -26.262231
DOT   Interoperability          0.170204  0.862069        -13.065941
NEAR           Layer 1          0.107516  0.586207        -27.866172
FET                 AI          0.061112  0.931034          6.210637
OP             Layer 2          0.031098  0.931034          3.199101
AAVE              DeFi          0.027682  0.931034          8.553332
PEPE              Meme          0.025977  0.827586         -7.133008
UNI               DeFi          0.020717  0.551724        -25.558463
SUI            Layer 1          0.020208  0.965517         21.132482


## 4. Louvain Community Detection & Sector Assortativity

We partition the correlation graph into unsupervised communities using the seeded Louvain modularity algorithm.
We then compute the **Normalized Mutual Information (NMI)** between the discovered communities and the external sector taxonomy:
$$\text{NMI}(C, S) = \frac{2 \cdot I(C; S)}{H(C) + H(S)}$$
to determine whether cryptocurrency market correlations reflect sector fundamentals (e.g. DeFi, Layer 1, Meme) or market-wide beta clustering.


In [5]:
detector = CommunityDetector(seed=42)
comm_res = detector.detect_communities(G_corr)

communities = comm_res["communities"]
node_comm = comm_res["node_community"]
modularity_score = comm_res["modularity"]

# Compare with sector labels
pred_labels = [node_comm.get(a, 0) for a in asset_names]
true_sectors = [sector_map.get(a, "Other") for a in asset_names]
unique_sec = sorted(list(set(true_sectors)))
sec_labels = [unique_sec.index(s) for s in true_sectors]

nmi = normalized_mutual_info_score(sec_labels, pred_labels)

print(f"Louvain Community Detection Results:")
print(f"  Number of Discovered Communities: {len(communities)}")
print(f"  Network Modularity Q:              {modularity_score:.4f}")
print(f"  NMI vs External Sectors:           {nmi:.4f}")

for c_idx, comm in enumerate(communities):
    coins = sorted(list(comm))
    sec_breakdown = [sector_map.get(c, "Other") for c in coins]
    print(f"  Community {c_idx + 1} ({len(coins)} coins): {', '.join(coins)} [Dominant: {max(set(sec_breakdown), key=sec_breakdown.count)}]")


Louvain Community Detection Results:
  Number of Discovered Communities: 4
  Network Modularity Q:              0.0223
  NMI vs External Sectors:           0.4091
  Community 1 (13 coins): AVAX, BCH, BNB, BTC, DOGE, ETC, ETH, LINK, PEPE, SHIB, SOL, XLM, XRP [Dominant: Layer 1]
  Community 2 (1 coins): TRX [Dominant: Layer 1]
  Community 3 (7 coins): APT, ATOM, DOT, FIL, ICP, LTC, RENDER [Dominant: DePIN]
  Community 4 (9 coins): AAVE, ADA, ARB, FET, INJ, NEAR, OP, SUI, UNI [Dominant: DeFi]


## 5. Crisis Topological Contraction: MST Normalized Tree Length

Under systemic market panic, asset correlations surge toward 1.0 (loss of diversification), causing the ultrametric distance $d_{ij} = \sqrt{2(1-\rho_{ij})}$ to collapse toward 0.
The Onnela et al. (2003) normalized tree length:
$$L(t) = \frac{1}{N - 1} \sum_{(i,j) \in \text{MST}} d_{ij}$$
serves as a pure topological crisis indicator. We load the causal trailing MST series from `network_features.pt` and overlay documented historical events.


In [6]:
net_feat_path = Path("../data/processed/v2/network_features.pt")
net_data = torch.load(net_feat_path, map_location="cpu", weights_only=False)

mst_series = pd.Series(net_data["mst_norm_length"], index=timestamps)
mod_series = pd.Series(net_data["modularity"], index=timestamps)

# Daily resample for smooth longitudinal visualization
mst_daily = mst_series.resample("1D").mean().dropna()

fig_mst = go.Figure()
fig_mst.add_trace(go.Scatter(
    x=mst_daily.index, y=mst_daily.values, mode="lines",
    name="MST Normalized Tree Length L(t)", line=dict(color="#ff9100", width=2)
))

# Overlay events within sample window
mst_tz = mst_daily.index.tz
for _, ev in events_df.iterrows():
    ev_date = pd.to_datetime(ev["date"])
    if mst_tz is not None:
        ev_date = ev_date.tz_localize(mst_tz)
    ev_label = str(ev.get("name", ev.get("event", "Event")))
    if mst_daily.index.min() <= ev_date <= mst_daily.index.max():
        fig_mst.add_vline(
            x=ev_date, line_dash="dot", line_color="#ef5350",
            annotation_text=ev_label[:18], annotation_position="top left"
        )

fig_mst.update_layout(
    title="MST Normalized Tree Length L(t) Across Historical Crises (Topological Compression)",
    xaxis_title="Date", yaxis_title="Normalized Tree Length (lower = higher market panic)",
    template="plotly_dark", height=450
)
fig_mst.show()


## 6. Independent Cascade Propagation Baseline

We inspect the performance of the classical probabilistic Independent Cascade (IC) diffusion baseline, fitted strictly on training cascades and evaluated across prediction horizons $k \in \{1, 3, 6, 12, 24\}$ hours on the validation and test splits (`reports/cascade_baseline.csv`).


In [7]:
cascade_rep_path = Path("../reports/cascade_baseline.csv")
if cascade_rep_path.exists():
    df_cascade = pd.read_csv(cascade_rep_path)
    print("=== INDEPENDENT CASCADE BENCHMARK PERFORMANCE ===")
    print(df_cascade[["horizon_hours", "val_auroc", "val_auprc", "test_auroc", "test_auprc"]].to_string(index=False))

    fig_casc = go.Figure()
    fig_casc.add_trace(go.Scatter(x=df_cascade["horizon_hours"], y=df_cascade["val_auprc"], name="Validation AUPRC", mode="lines+markers", line=dict(color="#00e676", width=2)))
    fig_casc.add_trace(go.Scatter(x=df_cascade["horizon_hours"], y=df_cascade["test_auprc"], name="Test AUPRC", mode="lines+markers", line=dict(color="#29b6f6", width=2)))
    fig_casc.update_layout(
        title="Independent Cascade Contagion Prediction AUPRC Across Horizons",
        xaxis_title="Propagation Horizon (Hours)", yaxis_title="Average Precision (AUPRC)",
        template="plotly_dark", height=400
    )
    fig_casc.show()
else:
    print("Run `python scripts/evaluate_cascade_baseline.py` to regenerate cascade benchmark reports.")


=== INDEPENDENT CASCADE BENCHMARK PERFORMANCE ===
 horizon_hours  val_auroc  val_auprc  test_auroc  test_auprc
             1   0.500019   0.004345         0.5    0.003614
             3   0.500019   0.010503         0.5    0.009921
             6   0.500000   0.018776         0.5    0.018430
            12   0.500000   0.033056         0.5    0.034364
            24   0.500000   0.064337         0.5    0.060153


## 7. Exported Artifacts & Econometric Rankings Verification

We verify the generated artifacts for downstream deep learning integration and TempME validation:
1. `data/processed/v2/rankings.csv`: Ground-truth econometric and structural influence rankings (DY NET, PageRank, Eigenvector) on the frozen 30-coin universe.
2. `data/processed/v2/network_features.pt`: Strictly causal trailing centralities, community IDs, and sector embeddings for TGN node feature augmentation.


In [8]:
rankings_path = Path("../data/processed/v2/rankings.csv")
if rankings_path.exists():
    rankings_df = pd.read_csv(rankings_path)
    print("Exported 30-Coin Influence Rankings (First 10 Rows):")
    print(rankings_df.head(10).to_string(index=False))

print(f"\nCausal Network Features Tensor:")
print(f"  Shape:     {tuple(net_data['network_features'].shape)} (T, N, F_net)")
print(f"  Features:  {net_data['feature_names']}")
print(f"  Zero NaNs: {not torch.isnan(net_data['network_features']).any()}")


Exported 30-Coin Influence Rankings (First 10 Rows):
coin  dy_net_spillover  pagerank  eigenvector  dy_rank  pagerank_rank  eigenvector_rank
 SOL         26.518108  0.058499     0.038728        1              4                 1
LINK         26.169465  0.008524     0.038626        2             29                 2
DOGE         22.270182  0.016446     0.037959        3             20                 3
 ETH         19.803326  0.040187     0.037603        4              8                 4
 SUI         19.089922  0.016229     0.037457        5             21                 5
 ADA         17.656277  0.030857     0.037225        6             12                 6
 XRP         15.265331  0.074419     0.036804        7              3                 7
 BTC         14.069719  0.024301     0.036649        8             15                 8
AVAX         11.321198  0.021055     0.036265        9             18                 9
 DOT         10.885812  0.058145     0.035945       10             

  Zero NaNs: True
